## Simulating traffic around a lock
In this notebook, we simulate a lock on a network which two opposingly directed vessels have to pass. We add a pre-coded complex lock object on the graph: only one vessel can be levelled at a time. The second vessel waits until the first vessel has passed the lock. 

#### 0. Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
import numpy as np

# import of modules important for locking
from opentnsim.lock import lock_new as lock_module
from opentnsim import vessel_traffic_service as vessel_traffic_service_module

# package(s) needed for inspecting the output
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.7


#### 1. Define object classes

In [2]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        lock_module.PassesLockComplex,             # allows to interact with a lock
        opentnsim.core.Identifiable,               # allows to give the object a name and a random ID,
        opentnsim.core.Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        opentnsim.core.VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        opentnsim.core.ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        opentnsim.graph.HasMultiDiGraph,           # allow to operate on a graph that can include parallel edges from and to the same nodes
        opentnsim.output.HasOutput,                # allow additional output to be stored
    ), 
    {}
)

#### 2. Create graph

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-350600,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(350600,0)))

# add edges
graph.add_edge('-1','0', weight=1)
graph.add_edge('0','-1', weight=1)
graph.add_edge('0','1', weight=1)
graph.add_edge('1','0', weight=1);
graph.add_edge('1','+1', weight=1)
graph.add_edge('+1','1', weight=1)

In [4]:
opentnsim.graph.plot_graph(graph)

#### 3. Run simulation

In [5]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [6]:
def generate_vessel(
    env,
    name,
    start_node,
    end_node,
    arrival_time,
    vessel_speed=4,
    vessel_length=100,
    vessel_beam=20,
    vessel_draft=10,
    vessel_type="tanker"
    ):
    
    # Ensure nodes are strings
    start_node = str(start_node)
    end_node = str(end_node)

    try:
        route = nx.dijkstra_path(env.graph, start_node, end_node)
    except nx.NetworkXNoPath:
        print(f"⚠️ No path from {start_node} to {end_node}. Vessel {name} not created.")
        return None

    geometry = env.graph.nodes[start_node]['geometry']

    data_vessel = {
        "env": env,
        "name": name,
        "geometry": geometry,
        "route": route,
        "v": vessel_speed,
        "L": vessel_length,
        "B": vessel_beam,
        "T": vessel_draft,
        "type": vessel_type,
        "arrival_time": arrival_time,
    }

    vessel = Vessel(**data_vessel)

    return vessel

In [7]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

# add graph to environment
env.graph = graph

# add components important for locking to the environment
env.vessel_traffic_service = vessel_traffic_service_module.VesselTrafficService(graph=graph)

lock = lock_module.IsLockComplex(
    env=env,
    name='Lock',
    node_open='0',
    node_A = '0',
    node_B = '1',
    distance_lock_doors_A_to_waiting_area_A = 4800,
    distance_lock_doors_B_to_waiting_area_B = 4800,
    distance_from_start_node_to_lock_doors_A = 4800,
    distance_from_end_node_to_lock_doors_B = 4800,
    lock_length = 400,
    lock_width = 50,
    lock_depth = 15,
    levelling_time = 300,
    sailing_distance_to_crossing_point = 1800,
    doors_opening_time= 300,
    doors_closing_time= 300,
    speed_reduction_factor_lock_chamber=0.5,
    sailing_in_time_gap_through_doors = 300,
    sailing_in_speed_sea = 1.5,
    sailing_in_speed_canal = 1.5,
    sailing_out_time_gap_through_doors = 120,
    sailing_time_before_opening_lock_doors = 600,
    sailing_time_before_closing_lock_doors = 120,
    registration_nodes = ['-1','+1'],
)

# # create vessels from dict 
# vessel_speed_outside_of_lock = 4. 

# data_vessel_in = {
#     "env": env,                                          # needed for simpy simulation
#     "name": "Vessel 1",                                  # required by Identifiable
#     "geometry": env.graph.nodes['0']['geometry'],        # required by Locatable
#     "route": nx.dijkstra_path(env.graph, "0", "1"),      # required by Routeable
#     "v": vessel_speed_outside_of_lock,                   # required by Movable, 4 m/s to check if the distance is covered in the expected time
#     "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
#     "B": 20,                                             # required by VesselProperties
#     "T": 10,                                             # required by VesselProperties
#     "type": 'tanker',                                    # required by VesselProperties
#     "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
# }  

# data_vessel_out = {
#     "env": env,                                          # needed for simpy simulation
#     "name": "Vessel 2",                                  # required by Identifiable
#     "geometry": env.graph.nodes['1']['geometry'],        # required by Locatable
#     "route": nx.dijkstra_path(env.graph, "1", "0"),      # required by Routeable
#     "v": vessel_speed_outside_of_lock,                   # required by Movable, 4 m/s to check if the distance is covered in the expected time
#     "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
#     "B": 20,                                             # required by VesselProperties
#     "T": 10,                                             # required by VesselProperties
#     "type": 'tanker',                                    # required by VesselProperties
#     "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
# }  

vessels=[]

arrival_time = pd.Timestamp('2025-01-01 01:00:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 1",
    start_node="-1",
    end_node="+1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 01:20:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 2",
    start_node="-1",
    end_node="+1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 02:00:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 3",
    start_node="-1",
    end_node="+1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 02:10:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 4",
    start_node="+1",
    end_node="-1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 02:15:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 5",
    start_node="+1",
    end_node="-1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 02:20:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 6",
    start_node="-1",
    end_node="+1",
    arrival_time=arrival_time
)
vessels.append(vessel)

arrival_time = pd.Timestamp('2025-01-01 03:20:00')
vessel = generate_vessel(
    env=env,
    name="Vessel 7",
    start_node="-1",
    end_node="+1",
    arrival_time=arrival_time
)
vessels.append(vessel)

for vessel in vessels:
    env.process(mission(env, vessel))
    

In [8]:
n_max = 4
vessel_speed_outside_of_lock = 4

# Part III, Ch3, Eq. 3.2 (NB: de helft van de looptime wordt hier effectief geimplementeerd door de sailing to lock te berekenen)
t_sailing_to_lock = lock.sailing_distance_to_crossing_point/vessel_speed_outside_of_lock
T_entering = t_sailing_to_lock + (n_max-1)*lock.sailing_in_time_gap_through_doors + 50/2

# Part III, Ch3, Eq. 3.3
T_operation = lock.doors_closing_time + lock.levelling_time + lock.doors_opening_time

# Part III, Ch3, Eq. 3.4 (NB: de helft van de looptime wordt hier effectief geimplementeerd door de sailing out of lock te berekenen)
t_sailing_out_of_lock = lock.sailing_distance_to_crossing_point/vessel_speed_outside_of_lock
T_exiting = t_sailing_out_of_lock + (n_max-1)*lock.sailing_out_time_gap_through_doors + 350/2

# Part III, Ch3, Eq. 3.1
T_locking = T_entering + T_operation + T_exiting
T_c = 2 * T_locking

C_s = 2*n_max / (T_c/3600)

print(f"The capacity of the lock is {np.round(C_s,1)} vessels per hour")

The capacity of the lock is 4.4 vessels per hour


In [9]:
IC = 1
T_arrival = T_locking*(1/IC)

In [10]:
predicted_arrival_moments = pd.date_range(simulation_start+pd.Timedelta(seconds=3000/4),simulation_start+pd.Timedelta(days=1),freq=pd.Timedelta(seconds=T_arrival))

In [11]:
# vessels = []
# vessel_index = 1
# for index,start_time in enumerate(predicted_arrival_moments):
#     start_time -= pd.Timedelta(seconds=3000/4)

#     new_arrival_time = start_time
#     if not index%2:
#         for index_vessel_in_operation in np.arange(1,5):     
#             data_vessel_in["arrival_time"] = new_arrival_time
#             vessel = Vessel(**data_vessel_in)
#             vessel.name = f'Vessel {vessel_index}'
#             env.process(mission(env, vessel))
#             new_arrival_time += pd.Timedelta(minutes=5)
#             vessels.append(vessel)
#             vessel_index += 1

#     else:
#         for index_vessel_in_operation in np.arange(1,5):     
#             data_vessel_out["arrival_time"] = new_arrival_time
#             vessel = Vessel(**data_vessel_out)
#             vessel.name = f'Vessel {vessel_index}'
#             env.process(mission(env, vessel))
#             new_arrival_time += pd.Timedelta(minutes=5)
#             vessels.append(vessel)
#             vessel_index += 1
            
#     if vessel_index >= 16:
#         break
        
env.run()

Vessel 1 2025-01-01 01:00:00 new
Vessel 2 2025-01-01 01:20:00 planned
Vessel 3 2025-01-01 02:00:00 new
Vessel 4 2025-01-01 02:10:00 new
Vessel 5 2025-01-01 02:15:00 planned
Vessel 6 2025-01-01 02:20:00 planned
Vessel 6 Vessel 4 0 days 00:11:31.231101178
0 days 00:05:40.172786470
Vessel 6 Vessel 5 0 days 00:11:31.231101178
Vessel 7 2025-01-01 03:20:00 new


#### 4. Inspect output

In [12]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock.logbook)

print("'{}' logbook data:".format(lock.name))  
print('')

display(lock_df)

'Lock' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock doors closing start,2025-01-02 01:44:03.000000,{},0
1,Lock doors closing stop,2025-01-02 01:49:03.000000,{},0
2,Lock chamber converting start,2025-01-02 01:49:03.000000,{},0
3,Lock chamber converting stop,2025-01-02 01:54:03.000000,{},1
4,Lock doors opening start,2025-01-02 01:54:03.000000,{},1
5,Lock doors opening stop,2025-01-02 01:59:03.000000,{},1
6,Lock doors closing start,2025-01-02 02:03:51.596112,{},1
7,Lock doors closing stop,2025-01-02 02:08:51.596112,{},1
8,Lock chamber converting start,2025-01-02 02:08:51.596112,{},1
9,Lock chamber converting stop,2025-01-02 02:13:51.596112,{},0


In [13]:
# We can plot the time-distance diagram
fig = lock.create_time_distance_plot(vessels = vessels, 
                                     xlimmin = -6050, 
                                     xlimmax = 6050,
                                     ylimmin = pd.Timestamp('2025-01-01 22:00:00'),
                                     ylimmax = pd.Timestamp('2025-01-02 09:00:00'),
                                     method='Plotly')
fig

In [14]:
lock.vessel_planning

,id,bound,L,B,T,operation_index,time_of_registration,time_of_acceptance,time_arrival_at_waiting_area,time_arrival_at_lineup_area,time_lock_passing_start,time_lock_entry_start,time_lock_entry_stop,time_lock_departure_start,time_lock_departure_stop,time_lock_passing_stop,delay,time_potential_lock_door_opening_stop,time_potential_lock_door_closure_start
0,83bb15c8-fcb4-40b8-90ed-d72294988076,0,100,20,10,0,2025-01-01 01:00:00,2025-01-01 01:00:00,2025-01-02 01:00:00,NaN,2025-01-02 01:12:30,2025-01-02 01:20:00,2025-01-02 01:25:40.172786470,2025-01-02 01:59:02.980561764,2025-01-02 01:59:51.576674116,2025-01-02 02:07:21.576674116,0 days 00:18:22.807775294,2025-01-02 01:10:00.000000000,2025-01-02 01:25:40.172786470
1,afecdfbb-3cec-4ac4-b7d0-95d9d2b09293,0,100,20,10,0,2025-01-01 01:20:00,2025-01-01 01:20:00,2025-01-02 01:20:00,NaN,2025-01-02 01:32:30,2025-01-02 01:40:00,2025-01-02 01:44:02.980561764,2025-01-02 01:59:25.788337058,2025-01-02 02:01:51.576674116,2025-01-02 02:09:21.576674116,0 days 00:00:00,2025-01-02 01:30:00.000000000,2025-01-02 01:44:02.980561764
2,3c2fa7c3-27af-4b63-b367-fbd7590d90a9,0,100,20,10,2,2025-01-01 02:00:00,2025-01-01 02:00:00,2025-01-02 02:00:00,NaN,2025-01-02 02:21:21.576674116,2025-01-02 02:28:51.576674116,2025-01-02 02:34:31.749460586,2025-01-02 02:59:02.980561764,2025-01-02 02:59:51.576674116,2025-01-02 03:07:21.576674116,0 days 00:18:22.807775294,2025-01-02 02:18:51.576674116,2025-01-02 02:34:31.749460586
3,8f47404b-5d3a-4112-b061-0f03aa0068ca,1,100,20,10,3,2025-01-01 02:10:00,2025-01-01 02:10:00,2025-01-02 02:10:00,NaT,2025-01-02 03:09:21.576674116,2025-01-02 03:16:51.576674116,2025-01-02 03:22:31.749460586,2025-01-02 03:41:34.730022350,2025-01-02 03:42:23.326134702,2025-01-02 03:49:53.326134702,0 days 00:50:54.557235880,2025-01-02 03:06:51.576674116,2025-01-02 03:22:31.749460586
4,59fd1634-0d02-4e98-ae84-03710f2346c9,1,100,20,10,3,2025-01-01 02:15:00,2025-01-01 02:15:00,2025-01-02 02:15:00,NaT,2025-01-02 03:15:01.749460586,2025-01-02 03:22:31.749460586,2025-01-02 03:26:34.730022350,2025-01-02 03:41:57.537797644,2025-01-02 03:44:23.326134702,2025-01-02 03:51:53.326134702,0 days 00:11:31.231101178,2025-01-02 03:12:31.749460586,2025-01-02 03:26:34.730022350
5,b3d5109b-954d-4e47-8f25-ee6a35db9701,0,100,20,10,2,2025-01-01 02:20:00,2025-01-01 02:20:00,2025-01-02 02:20:00,NaN,2025-01-02 02:32:30,2025-01-02 02:40:00,2025-01-02 02:44:02.980561764,2025-01-02 02:59:25.788337058,2025-01-02 03:01:51.576674116,2025-01-02 03:09:21.576674116,0 days 00:00:00,2025-01-02 02:30:00.000000000,2025-01-02 02:44:02.980561764
6,48507877-7ad3-4f5f-bacd-03bb8e7a082d,0,100,20,10,4,2025-01-01 03:20:00,2025-01-01 03:20:00,2025-01-02 03:20:00,NaN,2025-01-02 03:51:53.326134702,2025-01-02 03:59:23.326134702,2025-01-02 04:05:03.498921172,2025-01-02 04:20:03.498921172,2025-01-02 04:20:52.095033524,2025-01-02 04:28:22.095033524,0 days 00:19:23.326134702,2025-01-02 03:49:23.326134702,2025-01-02 04:05:03.498921172


In [15]:
lock.operation_planning

,bound,vessels,capacity_L,capacity_B,time_potential_lock_door_opening_stop,time_operation_start,time_entry_start,time_entry_stop,time_door_closing_start,time_door_closing_stop,...,time_door_opening_stop,time_departure_start,time_departure_stop,time_operation_stop,time_potential_lock_door_closure_start,wlev_A,wlev_B,maximum_individual_delay,total_delay,status
lock_operation,,,,,,,,,,,,,,,,,,,,,
0,0,[<__main__.Vessel object at 0x00000284A1B8AE00...,200,10,2025-01-02 01:10:00,2025-01-02 01:12:30,2025-01-02 01:20:00,2025-01-02 01:44:02.980561764,2025-01-02 01:44:02.980561764,2025-01-02 01:49:02.980561764,...,2025-01-02 01:59:02.980561764,2025-01-02 01:59:02.980561764,2025-01-02 02:01:51.576674116,2025-01-02 02:09:21.576674116,2025-01-02 02:03:51.576674116,NaN,NaN,0 days 00:18:22.807775294,0 days 00:18:22.807775294,unavailable
1,1,[],400,50,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:08:51.576674116,...,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,NaN,NaN,0 days 00:00:00,0 days 00:00:00,available
2,0,[<__main__.Vessel object at 0x00000284A1B4A680...,200,10,2025-01-02 02:18:51.576674116,2025-01-02 02:21:21.576674116,2025-01-02 02:28:51.576674116,2025-01-02 02:44:02.980561764,2025-01-02 02:44:02.980561764,2025-01-02 02:49:02.980561764,...,2025-01-02 02:59:02.980561764,2025-01-02 02:59:02.980561764,2025-01-02 03:01:51.576674116,2025-01-02 03:09:21.576674116,2025-01-02 03:03:51.576674116,NaN,NaN,0 days 00:18:22.807775294,0 days 00:18:22.807775294,unavailable
3,1,[<__main__.Vessel object at 0x00000284A1B4B160...,200,10,2025-01-02 03:06:51.576674116,2025-01-02 03:09:21.576674116,2025-01-02 03:16:51.576674116,2025-01-02 03:26:34.730022350,2025-01-02 03:26:34.730022350,2025-01-02 03:31:34.730022350,...,2025-01-02 03:41:34.730022350,2025-01-02 03:41:34.730022350,2025-01-02 03:44:23.326134702,2025-01-02 03:51:53.326134702,2025-01-02 03:46:23.326134702,NaN,NaN,0 days 00:50:54.557235880,0 days 01:02:25.788337058,unavailable
4,0,[<__main__.Vessel object at 0x00000284A1BD8CA0>],300,30,2025-01-02 03:49:23.326134702,2025-01-02 03:51:53.326134702,2025-01-02 03:59:23.326134702,2025-01-02 04:05:03.498921172,2025-01-02 04:05:03.498921172,2025-01-02 04:10:03.498921172,...,2025-01-02 04:20:03.498921172,2025-01-02 04:20:03.498921172,2025-01-02 04:20:52.095033524,2025-01-02 04:28:22.095033524,2025-01-02 04:22:52.095033524,NaN,NaN,0 days 00:19:23.326134702,0 days 00:19:23.326134702,unavailable


In [16]:
lock.operation_planning.apply(lambda x: [vessel.name for vessel in (x.vessels)], axis=1)

lock_operation
0    [Vessel 1, Vessel 2]
1                      []
2    [Vessel 3, Vessel 6]
3    [Vessel 4, Vessel 5]
4              [Vessel 7]
dtype: object

In [17]:
pd.DataFrame(vessels[0].logbook)

,Message,Timestamp,Value,Geometry
0,Sailing from node -1 to node 0 start,2025-01-01 01:00:00.000000,0,POINT (-3.149493386123042 0)
1,Sailing from node -1 to node 0 stop,2025-01-02 01:00:00.000000,345600.0,POINT (-0.0449157642059761 0)
2,Sailing from node 0 to node 1 start,2025-01-02 01:00:00.000000,345600.0,POINT (-0.0449157642059761 0)
3,Sailing to first lock doors start,2025-01-02 01:00:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
4,Sailing to first lock doors stop,2025-01-02 01:20:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
5,Sailing to position in lock start,2025-01-02 01:20:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
6,Sailing to position in lock stop,2025-01-02 01:25:40.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
7,Waiting for other vessels in lock start,2025-01-02 01:25:40.172786,{},POINT (0.0013474729261793 0)
8,Waiting for other vessels in lock stop,2025-01-02 01:44:02.980562,{},POINT (0.0013474729261793 0)
9,Levelling start,2025-01-02 01:44:02.980562,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)


In [18]:
delays = []
for vessel in vessels:
    vessel_df = pd.DataFrame(vessel.logbook)
    waiting_stop = vessel_df[vessel_df.Message == "Waiting stop"]
    if not waiting_stop.empty:
        delay = (waiting_stop.Timestamp-vessel.metadata["arrival_time"]).iloc[0]
    else:
        delay = pd.Timedelta(seconds=0)
    delays.append(delay)

In [19]:
print(f"The average vessel delay is {np.round(np.average(delays).total_seconds()/60,1)} minutes")

The average vessel delay is 0.0 minutes


In [20]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock])
generate_vessel_gantt_chart(df_eventtable)

#### Thinking of a way to analyse Intensity
'get_vessels_during_leveling':
- Identify locking cycles (by looking at the lock logbook)
- From the vessel list identify which vessels were in the lock during that locking cycle

'calculate_cycle_looptimes':
- Using the info derived from 'get_vessels_during_leveling' calculate looptimes

'calculate_detailed_cycle_time':
- using the info from 'get_vessels_during_leveling' and 'calculate_cycle_looptimes' calculate detailed cycle times

In [21]:
leveling_cycles = opentnsim.lock.logutils.get_vessels_during_leveling(lock, vessels)
looptimes_df = opentnsim.lock.logutils.calculate_cycle_looptimes(leveling_cycles, vessels)
Tc_df = opentnsim.lock.logutils.calculate_detailed_cycle_time(lock, vessels, leveling_cycles)

display(pd.DataFrame(leveling_cycles))
display(looptimes_df)
display(Tc_df)

,leveling_start,leveling_stop,vessels_present
0,2025-01-02 01:49:03.000000,2025-01-02 01:54:03.000000,"[Vessel 1, Vessel 2]"
1,2025-01-02 02:08:51.596112,2025-01-02 02:13:51.596112,[]
2,2025-01-02 02:49:03.000000,2025-01-02 02:54:03.000000,"[Vessel 3, Vessel 6]"
3,2025-01-02 03:31:35.000000,2025-01-02 03:36:35.000000,"[Vessel 4, Vessel 5]"
4,2025-01-02 04:10:03.498920,2025-01-02 04:15:03.498920,[Vessel 7]


,cycle,looptime_seconds
0,1,0.000000
1,2,NaN
2,3,NaN
3,4,899.980562
4,5,899.730022


,t_l_up,sum_t_i_up,T_close_up,T_waterlevel_up,T_open_up,sum_t_u_up,t_l_down,sum_t_i_down,T_close_down,T_waterlevel_down,T_open_down,sum_t_u_down,Tc_seconds,up_vessels,down_vessels,I_s
0,0,1442.980562,300.0,300.0,300.0,168.596112,0.000000,0.000000,300.0,300.0,300.0,0.000000,3411.576674,"[Vessel 1, Vessel 2]",[],2.110461
1,0,911.403888,300.0,300.0,300.0,168.596112,899.980562,583.153348,300.0,300.0,300.0,168.596112,4531.730022,"[Vessel 3, Vessel 6]","[Vessel 4, Vessel 5]",3.177594


In [22]:
for index, row in Tc_df.iterrows():
    print('Locking cycle {} has an intensity of {:.2f} vessels per hour'.format(index+1, row['I_s']))

Locking cycle 1 has an intensity of 2.11 vessels per hour
Locking cycle 2 has an intensity of 3.18 vessels per hour


In [23]:
lock.operation_planning

,bound,vessels,capacity_L,capacity_B,time_potential_lock_door_opening_stop,time_operation_start,time_entry_start,time_entry_stop,time_door_closing_start,time_door_closing_stop,...,time_door_opening_stop,time_departure_start,time_departure_stop,time_operation_stop,time_potential_lock_door_closure_start,wlev_A,wlev_B,maximum_individual_delay,total_delay,status
lock_operation,,,,,,,,,,,,,,,,,,,,,
0,0,[<__main__.Vessel object at 0x00000284A1B8AE00...,200,10,2025-01-02 01:10:00,2025-01-02 01:12:30,2025-01-02 01:20:00,2025-01-02 01:44:02.980561764,2025-01-02 01:44:02.980561764,2025-01-02 01:49:02.980561764,...,2025-01-02 01:59:02.980561764,2025-01-02 01:59:02.980561764,2025-01-02 02:01:51.576674116,2025-01-02 02:09:21.576674116,2025-01-02 02:03:51.576674116,NaN,NaN,0 days 00:18:22.807775294,0 days 00:18:22.807775294,unavailable
1,1,[],400,50,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:03:51.576674116,2025-01-02 02:08:51.576674116,...,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,2025-01-02 02:18:51.576674116,NaN,NaN,0 days 00:00:00,0 days 00:00:00,available
2,0,[<__main__.Vessel object at 0x00000284A1B4A680...,200,10,2025-01-02 02:18:51.576674116,2025-01-02 02:21:21.576674116,2025-01-02 02:28:51.576674116,2025-01-02 02:44:02.980561764,2025-01-02 02:44:02.980561764,2025-01-02 02:49:02.980561764,...,2025-01-02 02:59:02.980561764,2025-01-02 02:59:02.980561764,2025-01-02 03:01:51.576674116,2025-01-02 03:09:21.576674116,2025-01-02 03:03:51.576674116,NaN,NaN,0 days 00:18:22.807775294,0 days 00:18:22.807775294,unavailable
3,1,[<__main__.Vessel object at 0x00000284A1B4B160...,200,10,2025-01-02 03:06:51.576674116,2025-01-02 03:09:21.576674116,2025-01-02 03:16:51.576674116,2025-01-02 03:26:34.730022350,2025-01-02 03:26:34.730022350,2025-01-02 03:31:34.730022350,...,2025-01-02 03:41:34.730022350,2025-01-02 03:41:34.730022350,2025-01-02 03:44:23.326134702,2025-01-02 03:51:53.326134702,2025-01-02 03:46:23.326134702,NaN,NaN,0 days 00:50:54.557235880,0 days 01:02:25.788337058,unavailable
4,0,[<__main__.Vessel object at 0x00000284A1BD8CA0>],300,30,2025-01-02 03:49:23.326134702,2025-01-02 03:51:53.326134702,2025-01-02 03:59:23.326134702,2025-01-02 04:05:03.498921172,2025-01-02 04:05:03.498921172,2025-01-02 04:10:03.498921172,...,2025-01-02 04:20:03.498921172,2025-01-02 04:20:03.498921172,2025-01-02 04:20:52.095033524,2025-01-02 04:28:22.095033524,2025-01-02 04:22:52.095033524,NaN,NaN,0 days 00:19:23.326134702,0 days 00:19:23.326134702,unavailable
